# Sedov Case test

In [ ]:
from trustutils import run
run.introduction("K. Pons, W. Aboussi & E. Saikali")
run.TRUST_parameters()

## Description du Probleme:
Ce cas test de Sedov permet de tester la résolution des équations d'Euler compressible avec un gaz parfait pour une explosion cylindrique. De l'énergie est déposée au centre menant à une epansion violente. Une solution analytique est disponible en cylindrique et en sphérique. Ici seul le cas cylindrique est vérifié. 
Une sensibilité au choix de maillage est présentée pour un solveur type-Godunov.



In [ ]:

###########################################   Paramêtres du test   #################################################


####################################################
#Domaine et discretisation
##############################

#mesh_conf = {"1" , "2", "3", "4"}
mesh_conf = {"1" , "2"}
mesh_conf_x = {"1" : "28", "2" : "82", "3" : "244", "4" : "730"}
mesh_conf_y = {"1" : "28", "2" : "82", "3" : "244", "4" : "730"}

facsec = 0.4

####################################################

#
#thermo
gamma=1.4

#CI
E=1
rho0=1


In [ ]:
from trustutils import run
from IPython.display import display

comp= "cart"
number_of_partitions = {}

run.reset()
for msh in mesh_conf :
    nbcell = int(mesh_conf_x[msh]) * int(mesh_conf_y[msh])
    nx = int(mesh_conf_x[msh])
    ny = int(mesh_conf_y[msh])
                    
    name = f"{comp}_{nbcell}"
    number_of_partitions[name] = 8

    #----depot d'erj:
    v=1./(nx-1.)*1./(ny-1.)
    e = E / (rho0 * v)
    p_loc=(gamma - 1) * rho0 * e
    #----
            
    substitutions_dict = {"nx" : nx,
                            "ny" : ny,
                            "facsec" : facsec,
                            "npost" : int(nx/2),
                            "p_loc" : p_loc
                        }


    r = run.addCaseFromTemplate(f"jdd.data",targetDirectory=f"{name}",dic=substitutions_dict,nbProcs=number_of_partitions[name])
    if number_of_partitions[name] > 1:
        r.partition()

run.printCases()
run.runCases()
perf = run.tablePerf() # tableau des performances de calcul
display(perf)


In [ ]:
from trustutils import plot


instant = 0.1

#------------
# Recup valeurs dans fichier exterieur (resultats de "reference")
#--------
with open('src/cylindrical-sedov.out') as f1:
    for _ in range(19):
        next(f1)  # Ignore la ligne
        
    lines = f1.readlines()
    x_1 = [float(line.split()[1]) for line in lines]
    rho_1 = [float(line.split()[2]) for line in lines]
#--------


#------------
i=0;j=0
for msh in mesh_conf :            
    sonde = plot.Graph("", nX= 2,nY= 2)
    sonde.addPlot([i, j], "Numerical radial density at time "+str(instant)+" s " + "over x")
    nbcell = int(mesh_conf_x[msh]) * int(mesh_conf_y[msh])
    name = f"{comp}_{nbcell}"
    if comp == 'cart' : marker_type='-'
    sonde.addSegment(f"{name}/PAR_jdd_RHO_X.son", time=instant, label= f"gas {name}", marker=marker_type)
    sonde.add(x_1, rho_1, label = 'reference solution', marker='o')
    j=j+1
    sonde.label('x [m]', 'rho [$kg.m^{-3}$]')



#------------
#plot mesh convergence
sonde7 = plot.Graph("", nX= 1,nY= 1)
for msh in mesh_conf :
    nbcell = int(mesh_conf_x[msh]) * int(mesh_conf_y[msh])
    name = f"{comp}_{nbcell}"

    sonde7.addSegment(f"{name}/PAR_jdd_RHO_X.son", time=instant, label= f"gas {name}", marker='-')
    sonde7.add(x_1, rho_1, label = 'reference solution', marker='o')
        
sonde7.addPlot([0, 0], "Mesh convergence of density profile at time "+str(instant)+" s "+ "over x")
sonde7.label('y [m]', 'rho [$kg.m^{-3}$]')
#------------

#------------
for msh in mesh_conf :            
    sonde = plot.Graph("", nX= 1,nY= 1)
    sonde.addPlot([0, 0], "Numerical radial density at time "+str(instant)+" s " + "over x, y and vect=(0.5,0.5)")
    nbcell = int(mesh_conf_x[msh]) * int(mesh_conf_y[msh])
    name = f"{comp}_{nbcell}"
    if comp == 'cart' : marker_type='-'
    sonde.addSegment(f"{name}/PAR_jdd_RHO_X.son", time=instant, label= f"gas {name} x", marker=marker_type)
    sonde.addSegment(f"{name}/PAR_jdd_RHO_Y.son", time=instant, label= f"gas {name} y", marker=marker_type)
    sonde.addSegment(f"{name}/PAR_jdd_RHO_XY.son", time=instant, label= f"gas {name} xy", marker=marker_type)
    sonde.add(x_1, rho_1, label = 'reference solution', marker='o')

    sonde.label('x [m]', 'rho [$kg.m^{-3}$]')